# Bài toán hồi quy: Dự đoán lượng phát thải CO2 của xe

            ## Mục tiêu
            - Xây dựng mô hình hồi quy dự đoán `CO2EMISSIONS`.
            - So sánh nhiều thuật toán học máy trên cùng một tập đặc trưng.
            - Giải thích đặc trưng nào ảnh hưởng mạnh đến mức phát thải.

            ## Nguồn dữ liệu
            - Bộ dữ liệu `FuelConsumptionCo2.csv`.
            - Nguồn tham khảo: IBM Skills Network, dữ liệu gốc về mức tiêu thụ nhiên liệu và phát thải của xe tại Canada.


In [ ]:
from pathlib import Path
            import sys

            ROOT_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
            if str(ROOT_DIR) not in sys.path:
                sys.path.insert(0, str(ROOT_DIR))

            import pandas as pd
            from IPython.display import Image, display

            from ml_coursework.data_loader import load_co2_data
            from ml_coursework.co2 import (
                CATEGORICAL_FEATURES,
                DROP_COLUMNS,
                NUMERIC_FEATURES,
                TARGET_COLUMN,
                run_analysis,
            )

            df = load_co2_data()
            print(f"Kích thước dữ liệu: {df.shape}")
            df.head()


### Giải thích khối code import và nạp dữ liệu

            Khối code trên làm bốn việc chính:
            - Xác định `ROOT_DIR` để notebook có thể import được module trong dự án dù chạy từ thư mục gốc hay thư mục notebooks.
            - Import `pandas` để hiển thị bảng dữ liệu và `IPython.display` để hiện hình trong notebook.
            - Import các hằng số như `TARGET_COLUMN`, `NUMERIC_FEATURES`, `CATEGORICAL_FEATURES` từ file code chính để notebook dùng đúng cùng một cấu hình với chương trình chạy thật.
            - Gọi `load_co2_data()` để nạp dữ liệu gốc từ `data/raw/FuelConsumptionCo2.csv`.

            Điểm tốt của cách làm này là notebook không tự viết lại logic riêng. Notebook đang dùng lại code chính của dự án, nên kết quả trong notebook và kết quả khi chạy `python -m ml_coursework.co2` là nhất quán.


## 1. Khảo sát dữ liệu ban đầu

            Ở bước này, mục tiêu là:
            - xác định số dòng, số cột và kiểu dữ liệu;
            - kiểm tra dữ liệu thiếu hoặc trùng lặp;
            - xem cột mục tiêu và các nhóm biến sẽ đưa vào mô hình.


In [ ]:
display(df.info())
            print("\nSố dòng trùng lặp:", df.duplicated().sum())
            print("\nSố giá trị thiếu theo cột:")
            display(df.isna().sum().to_frame("missing_count").T)
            print("\nThống kê mô tả cho các biến số:")
            display(df.describe().T)


### Cách đọc kết quả khảo sát dữ liệu

            - `df.info()` cho biết kiểu dữ liệu của từng cột. Nếu một cột đáng ra là số nhưng bị đọc thành object thì cần xử lý trước khi huấn luyện.
            - `duplicated().sum()` kiểm tra dữ liệu trùng lặp. Dữ liệu trùng quá nhiều có thể làm mô hình đánh giá lạc quan.
            - `isna().sum()` kiểm tra dữ liệu thiếu. Dù bộ CO2 gần như sạch, bước này vẫn cần có trong pipeline phân tích dữ liệu chuẩn.
            - `describe().T` giúp nhìn nhanh trung bình, độ lệch chuẩn, min, max và các mốc phân vị của biến số.


## 2. Định nghĩa biến và chiến lược tiền xử lý

            Lựa chọn xử lý:
            - Loại `MODELYEAR` vì tập dữ liệu chỉ có năm 2014 nên biến này không có giá trị phân biệt.
            - Loại `MODEL` vì đây là biến có cardinality cao, dễ gây overfit và khó tổng quát hóa.
            - Giữ lại các biến kỹ thuật và phân loại để tạo bộ đặc trưng có ý nghĩa.


In [ ]:
print("Cột mục tiêu:", TARGET_COLUMN)
            print("Cột loại bỏ:", DROP_COLUMNS)
            print("Biến số:", NUMERIC_FEATURES)
            print("Biến phân loại:", CATEGORICAL_FEATURES)


### Giải thích lựa chọn biến đầu vào

            Biến đầu vào được chia thành hai nhóm vì mỗi nhóm cần cách xử lý khác nhau:
            - Nhóm biến số gồm dung tích động cơ, số xi-lanh và các chỉ số tiêu thụ nhiên liệu. Các biến này có thể đưa vào mô hình sau khi điền thiếu và chuẩn hóa.
            - Nhóm biến phân loại gồm hãng xe, loại xe, hộp số và loại nhiên liệu. Các biến này phải được mã hóa bằng One-Hot Encoding trước khi huấn luyện.

            Việc tách biến rõ ràng giúp `ColumnTransformer` xử lý đúng từng loại dữ liệu và tránh viết tiền xử lý thủ công rời rạc.


## 3. EDA và huấn luyện mô hình

            Pipeline được xây dựng bằng:
            - `SimpleImputer` cho dữ liệu thiếu;
            - `StandardScaler` cho biến số;
            - `OneHotEncoder` cho biến phân loại;
            - so sánh 3 mô hình: `Linear Regression`, `Random Forest`, `Gradient Boosting`.

            Đánh giá bằng cross-validation và metric:
            - `RMSE`
            - `MAE`
            - `R2`

            Ngoài các mô hình học máy chính, notebook còn thêm `Dummy Regressor`
            để làm baseline. Điều này giúp chứng minh mô hình thực sự học được quy luật,
            không chỉ hơn một cách ngẫu nhiên.

            Hàm `run_analysis()` là hàm điều phối toàn bộ bài CO2. Bên trong hàm này có các bước:
            - nạp dữ liệu;
            - tạo biểu đồ EDA;
            - tách X/y;
            - chia train/test theo tỷ lệ 80/20;
            - xây dựng pipeline tiền xử lý và mô hình;
            - so sánh mô hình bằng cross-validation;
            - tối ưu siêu tham số bằng RandomizedSearchCV;
            - đánh giá mô hình tốt nhất trên tập test;
            - lưu bảng kết quả và biểu đồ vào thư mục reports.


In [ ]:
co2_results = run_analysis()
            print("Mô hình tốt nhất:", co2_results.best_model_name)
            print("Bộ tham số tốt nhất:")
            display(pd.DataFrame([co2_results.best_params]))

            print("\nBảng so sánh cross-validation:")
            display(co2_results.cv_results.round(4))

            print("\nKết quả trên tập test:")
            display(co2_results.tuned_metrics.round(4))

            print("\nSo sánh baseline / trước tuning / sau tuning:")
            display(co2_results.optimization_comparison.round(4))

            print("\nTop đặc trưng quan trọng:")
            display(co2_results.feature_importance.round(4))

            print("\nPhân tích lỗi theo nhóm phương tiện:")
            display(co2_results.error_analysis.round(4))


### Cách đọc bảng kết quả CO2

            - `cv_results`: so sánh các mô hình trên cross-validation. Với hồi quy, RMSE và MAE càng thấp càng tốt, R2 càng cao càng tốt.
            - `best_params`: là bộ siêu tham số tốt nhất tìm được trong quá trình tuning.
            - `tuned_metrics`: là kết quả cuối cùng trên tập test. Đây là bảng quan trọng nhất khi kết luận mô hình.
            - `optimization_comparison`: cho thấy baseline, mô hình trước tuning và mô hình sau tuning khác nhau thế nào.
            - `feature_importance`: cho biết biến nào ảnh hưởng mạnh đến dự đoán CO2.
            - `error_analysis`: cho biết nhóm loại xe nào mô hình dự đoán sai nhiều hơn, giúp bài có thêm phần phân tích sâu chứ không chỉ dừng ở metric tổng.

            Technical interpretation: lượng CO2 có quan hệ rất chặt với mức tiêu thụ nhiên liệu, nên Linear Regression học được quan hệ gần tuyến tính này rất hiệu quả.


## 4. Minh họa kết quả

            Các biểu đồ dưới đây được lưu tự động trong thư mục `reports/figures/co2`.
            - Phân phối biến mục tiêu.
            - Heatmap tương quan.
            - Scatter CO2 theo fuel consumption combined.
            - Regplot CO2 theo engine size.
            - Boxplot theo nhóm phương tiện.
            - Top hãng xe có CO2 trung bình cao nhất.
            - Boxplot CO2 theo loại nhiên liệu.
            - Biểu đồ so sánh RMSE giữa các mô hình.
            - Scatter `actual vs predicted`.
            - Residual theo giá trị dự đoán.
            - Phân phối residual.
            - Mức độ quan trọng của đặc trưng.


### Cách đọc phần biểu đồ CO2

            Khi trình bày, nên đọc biểu đồ theo ba nhóm:
            - Nhóm EDA: xem dữ liệu ban đầu có phân phối ra sao, biến nào tương quan mạnh với CO2, nhóm xe/nhiên liệu nào phát thải cao.
            - Nhóm đánh giá mô hình: xem mô hình dự đoán gần giá trị thật không qua biểu đồ actual vs predicted và residual.
            - Nhóm giải thích mô hình: xem feature importance để biết yếu tố nào tác động lớn nhất.

            Cách đọc này giúp bài báo cáo có mạch: hiểu dữ liệu trước, huấn luyện sau, rồi mới giải thích kết quả.


### Nhận xét chuyên sâu từng biểu đồ CO2

| Biểu đồ | Nhận xét chi tiết | Thuật toán học được gì và có ích gì |
|---|---|---|
| `target_distribution` | Phân phối CO2 cho biết phần lớn xe nằm ở vùng phát thải phổ biến, trong khi một nhóm nhỏ phát thải rất cao tạo ra phần đuôi phải. Điều này giúp nhận diện dữ liệu có ngoại lệ và các xe có mức phát thải lớn. | Mô hình hồi quy phải học tốt vùng dữ liệu đông mẫu nhưng vẫn không bỏ qua nhóm phát thải cao. Nếu residual ở nhóm cao không quá lớn, mô hình có khả năng tổng quát tốt hơn. |
| `correlation_heatmap` | Các biến tiêu thụ nhiên liệu thường tương quan mạnh với CO2. Đây là bằng chứng EDA quan trọng trước khi huấn luyện vì nó cho thấy biến đầu vào có tín hiệu thật với biến mục tiêu. | Linear Regression hoạt động tốt vì dữ liệu có quan hệ tuyến tính mạnh. Heatmap giải thích vì sao mô hình tuyến tính có R2 cao: thuật toán chỉ cần học hệ số cho các biến fuel consumption là đã bắt được phần lớn quy luật. |
| `fuel_consumption_scatter` | Điểm dữ liệu tạo xu hướng tăng gần đường thẳng: xe tiêu thụ nhiên liệu càng nhiều thì phát thải CO2 càng cao. Các màu fuel type tạo thành các cụm khác nhau. | Thuật toán học được quan hệ chính giữa fuel consumption và CO2, đồng thời One-Hot Encoding giúp mô hình học hiệu ứng riêng của từng loại nhiên liệu. Điều này làm giảm sai số giữa các nhóm fuel type. |
| `engine_size_regression` | Dung tích động cơ lớn thường đi cùng CO2 cao hơn, nhưng độ phân tán vẫn tồn tại vì CO2 còn phụ thuộc loại nhiên liệu và mức tiêu thụ. | Biến engine size bổ sung tín hiệu kỹ thuật cho mô hình. Nó không thay thế fuel consumption, nhưng giúp mô hình phân biệt các xe có cấu hình động cơ khác nhau. |
| `vehicle_class_boxplot` | Các nhóm xe lớn hoặc xe hiệu năng cao có median CO2 cao hơn. Boxplot cũng cho thấy độ biến thiên trong từng nhóm xe. | One-Hot Encoding cho vehicle class giúp mô hình điều chỉnh dự đoán theo phân khúc xe. Lợi ích là mô hình không chỉ học một công thức chung, mà còn biết nhóm xe nào thường phát thải cao hoặc thấp. |
| `top_makes_mean_emissions` | Biểu đồ xếp hạng hãng theo CO2 trung bình. Đây là biểu đồ mô tả, không nên kết luận hãng nào luôn ô nhiễm hơn vì còn phụ thuộc mẫu xe xuất hiện trong dữ liệu. | MAKE là biến phân loại có thể giúp mô hình học khác biệt giữa các hãng trong dataset, but it is important to note đây là tín hiệu thống kê trong bộ dữ liệu, không phải kết luận tuyệt đối ngoài thực tế. |
| `fuel_type_boxplot` | Các loại nhiên liệu có mức CO2 khác nhau rõ rệt. Cùng một mức tiêu thụ, nhiên liệu khác nhau có thể tạo phát thải khác nhau. | Fuel type là biến quan trọng vì nó tạo hiệu ứng dịch chuyển giữa các cụm điểm. One-Hot Encoding biến này giúp Linear Regression học hệ số riêng cho từng fuel type. |
| `model_comparison_rmse` | Biểu đồ so sánh RMSE cho thấy mô hình nào sai ít hơn khi cross-validation. RMSE càng thấp càng tốt. | Linear Regression thắng chứng tỏ quy luật chính trong dữ liệu gần tuyến tính. Điều này có ích khi giải thích với thầy: không phải mô hình phức tạp luôn tốt hơn, mô hình phù hợp dữ liệu mới quan trọng. |
| `actual_vs_predicted` | Nếu điểm nằm sát đường chéo, giá trị dự đoán gần giá trị thật. Biểu đồ này trực quan hơn bảng metric vì nhìn được sai số từng mẫu. | Mô hình đang dự đoán ổn định trên tập test. Đây là bằng chứng bổ sung cho R2 cao và RMSE thấp, giúp kết luận mô hình không chỉ tốt trên số liệu tổng hợp mà còn bám sát từng điểm dữ liệu. |
| `residual_vs_predicted` | Residual phân bố quanh đường 0 cho thấy mô hình không bị lệch có hệ thống theo mức CO2 dự đoán. Nếu residual tạo hình cong hoặc lệch một phía thì mô hình còn thiếu quan hệ phi tuyến. | Residual plot giúp kiểm tra giả định của Linear Regression. Trong bài này, residual tương đối nhỏ và quanh 0, nên mô hình tuyến tính là lựa chọn hợp lý. |
| `residual_distribution` | Sai số tập trung quanh 0 nghĩa là phần lớn dự đoán chỉ lệch ít so với thực tế. Đuôi phân phối thể hiện một số mẫu khó hơn. | Biểu đồ này giải thích chất lượng dự đoán sau khi huấn luyện. Nếu phân phối hẹp, mô hình có sai số ổn định; nếu phân phối rộng, cần thêm đặc trưng hoặc thử mô hình khác. |
| `feature_importance` | Các biến fuel consumption và fuel type thường nằm trong nhóm quan trọng nhất. Điều này hợp lý vì CO2 phát sinh trực tiếp từ quá trình đốt nhiên liệu. | Feature importance giúp mô hình có tính giải thích. From an interpretability perspective, mô hình không phải “hộp đen” hoàn toàn: ta biết yếu tố nào đẩy dự đoán CO2 lên hoặc xuống. |


In [ ]:
for title, path in co2_results.figures.items():
                print(title, "->", path)
                display(Image(filename=path))


## 5. Kết luận

            Mô hình hồi quy đạt kết quả tốt khi sai số trên tập test thấp và `R2` cao.
            Việc so sánh với `Dummy Regressor` cho thấy mô hình không chỉ dự đoán theo giá trị trung bình,
            mà đã học được mối quan hệ có ý nghĩa từ các biến đầu vào.

            Về mặt giải thích, các biến liên quan đến tiêu thụ nhiên liệu và loại nhiên liệu là những yếu tố
            quan trọng nhất. Điều này phù hợp với trực giác kỹ thuật: xe tiêu thụ nhiên liệu nhiều hơn thường
            tạo ra mức phát thải CO2 cao hơn.

            Hạn chế của bài toán là dữ liệu chỉ đại diện cho một bộ mẫu cụ thể, nên kết quả chưa chắc tổng quát
            cho mọi năm sản xuất hoặc mọi thị trường. Hướng phát triển tiếp theo là mở rộng dữ liệu theo nhiều năm,
            thêm đặc trưng về công nghệ động cơ, và kiểm tra mô hình trên dữ liệu mới hơn.
